# Week 4 – PyTorch: training a neural network on MNIST, line by line

**The lab:** download MNIST and make DataLoaders (Task 1), define a network with 2 hidden layers (Task 2), write a training function (Task 3), an evaluation function (Task 4), and train for 10 epochs (Task 5).

**The secret:** this is the Week 3 pipeline again. Same six parts – PyTorch just does the boring maths (gradients) for you.

| Part | Week 3 (NumPy, by hand) | Week 4 (PyTorch) |
|---|---|---|
| Data | `X`, `y` arrays | `datasets.MNIST` + `DataLoader` |
| Model | `S = X @ W.T + b` | `nn.Linear` layers inside a class |
| Loss | softmax then `-log` | `nn.CrossEntropyLoss()` |
| Gradients | `dS = P - onehot`, chain rule by hand | `loss.backward()` |
| Update | `W -= lr * dW` | `optimizer.step()` |
| Evaluate | `argmax` == label | same |

## 0. Tensors – PyTorch's version of NumPy arrays
A **tensor** is a NumPy array that can (a) run on a GPU and (b) track gradients. Almost everything you learned about arrays carries over.

In [ ]:
import torch

t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(t.shape)              # torch.Size([2, 3])  – same idea as numpy .shape
print(t * 10)               # element-wise maths, like numpy
print(t.sum(), t.max())
print(t.argmax(dim=1))      # position of the max in each row -> tensor([2, 2])
print(t.numpy())            # convert to numpy
print(torch.zeros(2, 3), torch.rand(2))

`dim=1` means "along each row" (across the columns); `dim=0` means "down each column". You'll see `argmax(dim=1)` in the evaluation function: for each image (row), which class (column) got the highest score.

## 1. Imports and settings

In [ ]:
import torch
import torch.nn as nn                     # layers, loss functions
import torch.optim as optim               # optimisers (SGD, Adam)

from torch.utils.data import DataLoader   # serves data in batches
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

# Use a GPU if you have one, otherwise the CPU (the CPU is fine for MNIST)
if torch.cuda.is_available():
    device = "cuda"                       # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"                        # Apple Silicon Mac
else:
    device = "cpu"
print("using", device)

batch_size = 64                           # how many images per training step
torch.manual_seed(0)                      # make results repeatable

## 2. Task 1 – Data: datasets, transforms and DataLoaders

### Transforms: preparing each image
* `transforms.ToTensor()` – turns the 28×28 image (pixel values 0–255) into a tensor of shape **(1, 28, 28)** with values **0.0–1.0**. The `1` is the number of colour channels (greyscale = 1).
* `transforms.Normalize((0.1307,), (0.3081,))` – subtracts 0.1307 (the average MNIST pixel) and divides by 0.3081 (the standard deviation). Values end up centred around 0, which helps training. The numbers are just MNIST's mean and std – you don't need to memorise them.
* `transforms.Compose([...])` – do them in order.

> ⚠️ **Change the folder!** The lab's solution uses `root="/Users/shirindora/.datasets"` – that's the lecturer's own laptop and will fail (or do something weird) on yours. Use `"./data"`, which means "a `data` folder next to this notebook". If you ran Week 2, it's already downloaded.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

image, label = train_dataset[0]
print("one image:", image.shape, "| label:", label)
print(f"pixel range after normalising: {image.min():.2f} to {image.max():.2f}")

### DataLoader: serving the data in batches
Training on all 60,000 images at once is slow and memory-hungry; one image at a time is noisy. So we use **mini-batches** of 64.
* `shuffle=True` for training: a different random order each epoch, so the model doesn't learn the order.
* `shuffle=False` for testing: order doesn't matter there.

A `DataLoader` is something you **loop over**; each loop gives you one batch `(images, labels)`.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print("batches per epoch:", len(train_loader))      # 60000 / 64 ≈ 938

images, labels = next(iter(train_loader))            # grab ONE batch to inspect
print("images:", images.shape)    # [64, 1, 28, 28] = 64 images, 1 channel, 28x28
print("labels:", labels.shape)    # [64]            = 64 answers
print("first 10 labels:", labels[:10])

**Read shapes like a sentence:** `[64, 1, 28, 28]` = "64 images, each with 1 channel, 28 tall, 28 wide". Being able to do this is half the battle in PyTorch.

## 3. Task 2 – The model
### What's a layer?
`nn.Linear(in_features, out_features)` is **exactly** Week 3's $s = Wx + b$. It owns a weight matrix $W$ of shape (out, in) and a bias $b$ of shape (out). `nn.Linear(784, 512)` takes 784 numbers in and gives 512 numbers out.

### Why ReLU?
If you stack Linear layers directly, the result is still just one big linear function (a matrix times a matrix is a matrix) – no more powerful than Week 3. **ReLU** ($\max(0, x)$: keep positives, turn negatives into 0) between them adds a bend, which lets the network learn curvy, complicated patterns. Those in-between layers are called **hidden layers**.

### The network
```
image (1x28x28) --flatten--> 784 --Linear--> 512 --ReLU--> --Linear--> 256 --ReLU--> --Linear--> 10 scores
```
* `__init__` creates the layers (the pieces).
* `forward` says how data flows through them. You never call `forward` yourself – writing `model(images)` calls it for you.
* `torch.flatten(x, start_dim=1)` turns `[64, 1, 28, 28]` into `[64, 784]`: keep the batch dimension (dim 0), squash everything after it into one row of 784 pixels.

> **Note:** the task says 512, 256 and 10 neurons, but the lab's *solution* uses 256, 128 and 10. Both work almost identically. Below follows the task wording; change the numbers to compare.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()                     # let nn.Module set itself up (Week 1, classes!)

        self.fc1 = nn.Linear(28 * 28, 512)     # hidden layer 1: 784 -> 512   ("fc" = fully connected)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(512, 256)         # hidden layer 2: 512 -> 256
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(256, 10)          # output layer: 256 -> 10 scores (one per digit)

    def forward(self, x):                      # x: [batch, 1, 28, 28]
        x = torch.flatten(x, start_dim=1)      # -> [batch, 784]
        x = self.relu1(self.fc1(x))            # -> [batch, 512]
        x = self.relu2(self.fc2(x))            # -> [batch, 256]
        x = self.fc3(x)                        # -> [batch, 10]  raw scores ("logits")
        return x                               # no softmax here: the loss function does it

model = NeuralNetwork().to(device)             # .to(device) moves the weights to CPU/GPU
print(model)

In [ ]:
# Sanity checks: push one batch through and look at the shapes
out = model(images.to(device))
print("output shape:", out.shape)              # [64, 10]: 10 scores for each of 64 images

# How many numbers (weights + biases) will be learned?
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")
# fc1: 784*512 + 512, fc2: 512*256 + 256, fc3: 256*10 + 10

## 4. Task 3 – Loss, optimiser and the training function
* `nn.CrossEntropyLoss()` – softmax + $-\ln P(\text{correct})$, averaged over the batch. **Exactly Week 3.** That's why the model returns raw scores: the loss applies softmax itself.
* `optim.SGD(model.parameters(), lr=1e-3)` – stochastic gradient descent: $w \leftarrow w - lr \times grad$ for every parameter. `model.parameters()` hands it all the weights to update. `1e-3` = 0.001.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=1e-3)

### The training function, line by line
For **every batch**, the same five steps (memorise these – they're in every PyTorch project ever):

1. `optimizer.zero_grad()` – wipe old gradients. PyTorch *adds* new gradients to old ones by default, so forgetting this breaks training.
2. `outputs = model(images)` – **forward pass**: compute scores.
3. `loss = criterion(outputs, labels)` – how wrong are we?
4. `loss.backward()` – **backpropagation**: compute the gradient for every weight (Week 3's chain rule, automated).
5. `optimizer.step()` – update every weight a little bit downhill.

Other bits:
* `model.train()` – puts the model in "training mode". Doesn't matter for this simple network but matters for some layer types (dropout, batch norm), so always include it.
* `loss.item()` – turns a one-number tensor into a normal Python number.
* Returns the **average loss** over all batches in the epoch.

In [ ]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for images, labels in loader:                          # one batch at a time
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()                              # 1. clear old gradients
        outputs = model(images)                            # 2. forward pass  -> [64, 10]
        loss = criterion(outputs, labels)                  # 3. compute loss  -> one number
        loss.backward()                                    # 4. backprop: gradients for all weights
        optimizer.step()                                   # 5. update weights

        total_loss += loss.item()

    return total_loss / len(loader)                        # average loss per batch

## 5. Task 4 – The evaluation function
* `model.eval()` – "evaluation mode" (the opposite of `model.train()`).
* `with torch.no_grad():` – don't track gradients; we aren't learning here, so this saves memory and time.
* `outputs.argmax(dim=1)` – for each image, the class with the highest score = the prediction.
* `(preds == labels)` – a tensor of True/False; `.sum().item()` counts the Trues.
* Accuracy = correct / total.

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)                  # predicted digit for each image

            correct += (preds == labels).sum().item()
            total += labels.size(0)                        # batch size (last batch can be smaller)

    return correct / total

print(f"accuracy BEFORE training: {evaluate(model, test_loader):.4f}")   # ~0.1 = random guessing

## 6. Task 5 – Train for 10 epochs
An **epoch** = one full pass through all 60,000 training images (≈ 938 batches). Each epoch we train, then measure accuracy on the 10,000 test images the model has never trained on (Week 2: always judge on unseen data).

On a laptop CPU each epoch takes roughly 10–30 seconds. If you're short of time, set `epochs = 3`.

In [ ]:
epochs = 10
history = []

for epoch in range(1, epochs + 1):
    train_loss = train(model, train_loader, optimizer, criterion)
    test_acc = evaluate(model, test_loader)
    history.append((train_loss, test_acc))
    print(f"Epoch {epoch:02d} | Loss: {train_loss:.4f} | Test Acc: {test_acc:.4f}")

### Reading the results
* **Epoch 1 loss ≈ 2.2** – close to $\ln(10) = 2.30$, the random-guessing loss from Week 3. The network starts knowing nothing.
* Loss falls and accuracy rises to roughly **90%** after 10 epochs.
* It's still improving at the end: `lr=1e-3` with plain SGD is a **small** learning rate, so learning is slow (remember the Week 3 `lr=0.001` experiment).

In [ ]:
losses = [h[0] for h in history]; accs = [h[1] for h in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(range(1, epochs + 1), losses, 'o-'); ax1.set_title("training loss"); ax1.set_xlabel("epoch"); ax1.grid()
ax2.plot(range(1, epochs + 1), accs, 'o-');   ax2.set_title("test accuracy"); ax2.set_xlabel("epoch"); ax2.grid()
plt.show()

## 7. Look at what the model does
Numbers are nice, but look at actual predictions. Red titles are mistakes.

In [ ]:
images, labels = next(iter(test_loader))
model.eval()
with torch.no_grad():
    preds = model(images.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray")                 # images[i] is [1,28,28]; [0] takes the single channel
    right = preds[i] == labels[i]
    ax.set_title(f"pred {preds[i].item()}", color="green" if right else "red")
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Find some mistakes across the whole test set
wrong_imgs, wrong_pred, wrong_true = [], [], []
with torch.no_grad():
    for imgs, labs in test_loader:
        p = model(imgs.to(device)).argmax(dim=1).cpu()
        mask = p != labs
        wrong_imgs += list(imgs[mask]); wrong_pred += list(p[mask]); wrong_true += list(labs[mask])

print("number of mistakes:", len(wrong_imgs), "out of", len(test_dataset))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax in axes:
    ax.axis("off")
for ax, img, said, was in zip(axes, wrong_imgs, wrong_pred, wrong_true):   # zip pairs items up; stops at the shortest
    ax.imshow(img[0], cmap="gray")
    ax.set_title(f"said {said.item()}, was {was.item()}", fontsize=8)
plt.show()

## 8. Experiment: make it better
Change **one** thing (the "when would you change each part?" question from Week 3) and see what happens. Here we swap the optimiser for **Adam**, which adapts the step size for each weight and usually learns much faster. Everything else stays identical.

In [ ]:
model2 = NeuralNetwork().to(device)                          # fresh, untrained network
optimizer2 = optim.Adam(model2.parameters(), lr=1e-3)

for epoch in range(1, 4):                                    # only 3 epochs
    loss = train(model2, train_loader, optimizer2, criterion)
    acc = evaluate(model2, test_loader)
    print(f"Adam epoch {epoch} | Loss: {loss:.4f} | Test Acc: {acc:.4f}")

Adam reaches ~97% in 3 epochs, versus ~90% after 10 epochs of SGD at `lr=1e-3`.

### Try it yourself (one at a time)
1. Keep SGD but use `lr=0.1`. Faster? Try `lr=1.0` – does it break?
2. Use the solution's sizes (256, 128) instead of (512, 256). Does accuracy change much?
3. Remove the ReLUs. What happens to accuracy and why? (Hint: Week 3 – without ReLU the network is just one linear classifier.)
4. Change `batch_size` to 16 or 256. What happens to speed and accuracy?

### Saving your trained model (handy for coursework)
```python
torch.save(model.state_dict(), "mnist_model.pt")           # save the learned weights
model = NeuralNetwork(); model.load_state_dict(torch.load("mnist_model.pt"))   # load them back
```

## Week 4 checklist
- I can read tensor shapes like `[64, 1, 28, 28]` and `[64, 10]`.
- I know `ToTensor`/`Normalize` prepare images and `DataLoader` serves batches.
- I can write a network class with `__init__` (layers) and `forward` (data flow), and know `nn.Linear` is $Wx + b$.
- I can write the 5-step training loop from memory: `zero_grad`, forward, loss, `backward`, `step`.
- I can write an evaluation loop with `eval()`, `no_grad()`, `argmax`, and compute accuracy.
- I can explain why the first loss is ≈ 2.3 and why we test on unseen data.